<a href="https://colab.research.google.com/github/minh121-hub/medical-biostat-ai-portfolio/blob/main/01_logistic_regression_separation/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import pandas as pd
# 랜덤으로 변수 설정
np.random.seed(42)

age = np.random.normal(55,12,150).clip(30,80) # 정규분포
bmi = np.random.normal(26,4,150).clip(18,40)
smoking = np.random.binomial(1,0.3,150) # 이항분포
systolic_bp = np.random.normal(130,15,150).clip(90,180)

df = pd.DataFrame({'age':age,
                   'bmi':bmi,
                   'smoking':smoking,
                   'systolic_bp':systolic_bp})



In [22]:
def sigmoid(x):
  return 1/(1+np.exp(-x))

# 로지스틱과 시그모이드로 결과변수 생성
linear_comb = (0.03*age + 0.05*bmi + 0.8*smoking + 0.04*systolic_bp - 8)
prob = sigmoid(linear_comb)
outcome = np.random.binomial(1,prob)

df['outcome']=outcome

# 완전분리 상황을 의도적으로 생성(문제 상황 삽입)
# outcome과 일치하는 인공적 변수 삽입하여 완전분리 상황 유도
separator = df['outcome'].copy()
df['separator'] = separator

In [23]:
#완전분리 발생 확인(돌려보면 에러 뜸)
import statsmodels.api as sm

X = df[['age','bmi','smoking','systolic_bp','separator']]
X = sm.add_constant(X) # 계수를 추정할 열 추가
y = df['outcome']

model = sm.Logit(y, X).fit()
print(model.summary())

         Current function value: inf
         Iterations: 35


/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packa

LinAlgError: Singular matrix

In [27]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF는 X끼리의 상관관계(다중공선성), add_constant 포함해서 계산
X_vif = df[['age', 'bmi', 'smoking', 'systolic_bp', 'separator']]
X_vif = sm.add_constant(X_vif) # const의 vif는 원래 크게 나옴

vif_data = pd.DataFrame()
vif_data['variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)
# VIF 결과: 다중공신성 문제 아님

# 각 변수와 outcome 간 상관관계 (X와 y의 상관관계)
correlations = df[['age', 'bmi', 'smoking', 'systolic_bp', 'separator']].corrwith(df['outcome'])
print(correlations)
# 결과: separator의 상관관계가 1, 완전분리 확인

      variable         VIF
0        const  147.861871
1          age    1.052258
2          bmi    1.061294
3      smoking    1.110276
4  systolic_bp    1.123255
5    separator    1.087159
age            0.121079
bmi           -0.042219
smoking        0.151925
systolic_bp    0.133114
separator      1.000000
dtype: float64


In [28]:
# X에 'separator'빼고 재적합

X = df[['age','bmi','smoking','systolic_bp']]
X = sm.add_constant(X) # 계수를 추정할 열 추가
y = df['outcome']

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.619389
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                  150
Model:                          Logit   Df Residuals:                      145
Method:                           MLE   Df Model:                            4
Date:                Sun, 26 Jul 2026   Pseudo R-squ.:                 0.06253
Time:                        10:32:16   Log-Likelihood:                -92.908
converged:                       True   LL-Null:                       -99.106
Covariance Type:            nonrobust   LLR p-value:                   0.01464
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -3.8151      2.166     -1.761      0.078      -8.061       0.431
age             0.0302    